<a href="https://colab.research.google.com/github/http-hades/FUNDAI-Laboratories-WAPER/blob/main/Lab4_logic_KR_WAPER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Logic and Knowledge Representation

## Fundamentals of Artificial Intelligence

**Name:** [Jan Andrie C. Waper]

**Course:** [BSCS-AI 2]

**Section:** [09282 FUNDAI]

**Date:** [September 15, 2026]

**GitHub URL:** https://github.com/http-hades/FUNDAI-Laboratories-WAPER.git

## Description
This laboratory uses Python and SymPy to perform truth table generation, satisfiability checking, theorem proving, and logical education.




In [1]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, satisfiable
from itertools import product

In [2]:
P, Q, R = symbols('P Q R')

In [3]:
def print_truth_table(expression, symbol_list):
    header = [str(s) for s in symbol_list] + [str(expression)]
    print(" | ".join(header))
    print("-" * (5 * len(header)))

    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        result = expression.subs(mapping)
        row = [str(v) for v in values] + [str(result)]
        print(" | ".join(row))

    print()

In [4]:
print("Negation: NOT P")
print_truth_table(Not(P), [P])

print("Conjunction: P AND Q")
print_truth_table(And(P, Q), [P, Q])

print("Disjunction: P OR Q")
print_truth_table(Or(P, Q), [P, Q])

print("Implication: P -> Q")
print_truth_table(Implies(P, Q), [P, Q])

print("Biconditional: P <-> Q")
print_truth_table(Equivalent(P, Q), [P, Q])

Negation: NOT P
P | ~P
----------
False | True
True | False

Conjunction: P AND Q
P | Q | P & Q
---------------
False | False | False
False | True | False
True | False | False
True | True | True

Disjunction: P OR Q
P | Q | P | Q
---------------
False | False | False
False | True | True
True | False | True
True | True | True

Implication: P -> Q
P | Q | Implies(P, Q)
---------------
False | False | True
False | True | True
True | False | False
True | True | True

Biconditional: P <-> Q
P | Q | Equivalent(P, Q)
---------------
False | False | True
False | True | False
True | False | False
True | True | True



In [5]:
def is_tautology(expression, symbol_list):
    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        if not bool(expression.subs(mapping)):
            return False
    return True


law_of_excluded_middle = Or(P, Not(P))
contradiction = And(P, Not(P))
simple_implication = Implies(P, Q)

print("P OR NOT P is a tautology:", is_tautology(law_of_excluded_middle, [P]))
print("P AND NOT P is a tautology:", is_tautology(contradiction, [P]))
print("P -> Q is a tautology:", is_tautology(simple_implication, [P, Q]))

P OR NOT P is a tautology: True
P AND NOT P is a tautology: False
P -> Q is a tautology: False


In [6]:
def is_satisfiable(expression):
    return satisfiable(expression) is not False


print("P AND NOT P is satisfiable:", is_satisfiable(And(P, Not(P))))
print("P OR Q is satisfiable:", is_satisfiable(Or(P, Q)))
print("P -> Q is satisfiable:", is_satisfiable(Implies(P, Q)))

P AND NOT P is satisfiable: False
P OR Q is satisfiable: True
P -> Q is satisfiable: True


In [7]:
def to_conjuction(kb):
    if isinstance(kb, list):
        return And(*kb)
    return kb


def kb_entails(kb, conclusion):
    kb_expression = to_conjuction(kb)
    counter_check = And(kb_expression, Not(conclusion))
    return satisfiable(counter_check) is False


def check_entailment(kb, conclusion, label="Query"):
    holds = kb_entails(kb, conclusion)
    kb_expression = to_conjuction(kb)
    counterexample = satisfiable(And(kb_expression, Not(conclusion)))

    print(label)

    if holds:
        print("Result: Entailment holds.")
    else:
        print("Result Entailment does not hold.")
        print("Counterexample Model:", counterexample)

    print("_" * 60)
    return holds

In [8]:
Rain, Wet = symbols('Rain Wet')

kb_rain = [
    Implies(Rain, Wet),
    Rain
]

check_entailment(kb_rain, Wet, "Theorem Proving: Rain example")

Theorem Proving: Rain example
Result: Entailment holds.
____________________________________________________________


True

In [9]:
kb_invalid = [
    Implies(Rain, Wet),
    Wet
]

check_entailment(kb_invalid, Rain, "Invalid Inference: Affirming the consequent")

Invalid Inference: Affirming the consequent
Result Entailment does not hold.
Counterexample Model: {Wet: True, Rain: False}
____________________________________________________________


False

In [10]:
print("Logical Dedction Rules")
print("=" * 60)

#Modus Ponens
check_entailment(
    [P, Implies(P, Q)],
    Q,
    "Modus Ponens: P, P ->Q, therefore Q"
)

#Modus Tollens
check_entailment(
    [Not(Q), Implies(P,Q)],
    Not(P),
    "Modus Tollens: Not Q, P -> Q, therefore NOT P"
)

Logical Dedction Rules
Modus Ponens: P, P ->Q, therefore Q
Result: Entailment holds.
____________________________________________________________
Modus Tollens: Not Q, P -> Q, therefore NOT P
Result: Entailment holds.
____________________________________________________________


True

## Grounded First-Order Logic Example

Full First-Order Logic includes objects and quantifiers.
For this laboratory, we demonstrate a simple grounded FOL example by converting FOL atoms into propositional symbols.

English:

- All humans are mortal.
- Socrates is human.
- Therefore, Socrates is mortal.

Grounded propositional form:

- Human_Socrates -> Mortal_Socrates

In [11]:
Human_Socrates, Mortal_Socrates = symbols('Human_Socrates Mortal_Socrates')

kb_socrates = [
    Implies(Human_Socrates, Mortal_Socrates),
    Human_Socrates
]

check_entailment(
    kb_socrates,
    Mortal_Socrates,
    "Grounded FOL: Socrates is mortal"
)

Grounded FOL: Socrates is mortal
Result: Entailment holds.
____________________________________________________________


True

In [12]:
def make_human_mortal_kb(constants):
    kb = []
    human = {}
    mortal = {}

    for name in constants:
        h, m = symbols(f'Human_{name} Mortal_{name}')
        human[name] = h
        mortal[name] = m
        kb.append(Implies(h, m))

    return kb, human, mortal

constants = ["Socrates", "Plato"]

kb_people, human, mortal = make_human_mortal_kb(constants)

# Add facts
kb_people.append(human["Socrates"])
kb_people.append(human["Plato"])

# Query: Is Plato mortal?
check_entailment(
    kb_people,
    mortal["Plato"],
    "Grounded FOL with multiple constants: Is Plato mortal?"
)

Grounded FOL with multiple constants: Is Plato mortal?
Result: Entailment holds.
____________________________________________________________


True

## Guide Questions and Answers

### 1. What is the difference between syntax and semantics?

**Answer:** Syntax refers to the formal rules and structure used to construct valid statements in logic or programming. Semantics defines the actual meaning or truth values assigned to those validly formatted statements.

### 2. Why is `P -> Q` true when `P` is false?

**Answer:** In formal logic, a material implication `P -> Q` only promises that if `P` holds, `Q` must follow. When `P` is false, the condition is never met, making the implication vacuously true regardless of `Q`'s value.

### 3. What does it mean for a knowledge base to entail a conclusion?

**Answer:** A knowledge base entails a conclusion if the conclusion is logically guaranteed to be true whenever all sentences in the knowledge base are true. In other words, there is no possible model where the knowledge base holds true but the conclusion is false.

### 4. How does theorem proving use satisfiability checking?

**Answer:** Theorem proving checks if a conclusion follows from a knowledge base KB by proving that KB is unsatisfiable. If no model makes the knowledge base true and the negation of the query true at the same time, the entailment holds.

### 5. What is one limitation of propositional logic compared to First-Order Logic?

**Answer:** Propositional logic treats statements as atomic units, lacking the ability to directly express individual objects, relationships, or general quantifiers like "for all" and "there exists". This requires explicitly repeating rules for every single entity in the domain.

## Reflection

### Challenges Encountered

* Manually mapping every domain object to propositional symbols in grounded FOL gets tedious and scale-inefficient very quickly.

### What I Learned

* Grounding allows us to convert First-Order Logic into propositional logic so standard inference engines can check entailment for specific entities.